# Stage C — T4 horizon-3 handoff
Asserts a T4 and validates CPU/CUDA attention plus one full-geometry FP32/FP16 horizon-3 step. The extended capacity matrix is an A100 task.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='cc4ba30c153b0d029124c72292d364ca0963064c'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
RUN_NAME='c5_t4_horizon3'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess, sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
output=f'{DRIVE_ROOT}/runs/{RUN_NAME}'
Path(output).mkdir(parents=True,exist_ok=True)
def logged(label,command):
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',label,'--repo',str(repo),'--',*command],check=True)
logged('hardware_preflight',['seqtrainer-titans-stage-c-hardware-preflight','--require','T4','--output',f'{output}/hardware.json'])
logged('stage_c_tests',[sys.executable,'-m','pytest','-q',str(repo/'tests/test_titans_paper_mac_stage_c_tokenizers.py'),str(repo/'tests/test_titans_paper_mac_stage_c_model.py'),str(repo/'tests/test_titans_paper_mac_stage_c_smoke.py')])
logged('t4_gpu_smoke',['seqtrainer-titans-stage-c-gpu-smoke','--dataset-dir',DATASET_DIR,'--output',f'{output}/gpu_smoke.json','--require','T4'])

In [ ]:
# Colab T4 terminates a second long-lived full-geometry capacity process.
# Preserve the passed smoke evidence and defer multi-step/checkpoint/throughput capacity to Notebook 02 on A100.
logged('t4_bounded_evidence',['seqtrainer-titans-stage-c-t4-evidence','--smoke',f'{output}/gpu_smoke.json','--output-dir',output])
print('SHARE THIS DIRECTORY:',output)